In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt

# Load and prep active users

df2 = pd.read_excel(
    "../mock-data/Active_Inactive_Non-Diala_Mock.xlsx",
    sheet_name="Active_Users"
)
df2 = df2.dropna(how="all")
print(f"Active raw shape: {df2.shape}")

# Numeric coerce
df2["Diab_Duration"] = pd.to_numeric(df2["Diab_Duration"], errors="coerce")
for col in ["AGE", "AOS", "Diab_Duration", "HbA1c_BL", "HbA1c_FU", "Kupp_Occupation"]:
    if col in df2.columns:
        df2[col] = pd.to_numeric(df2[col], errors="coerce")

# String clean
replace_vals = ["", " ", "NA", "N/A", "na", "n/a", "NIL", "nil",
                "None", "none", "NULL", "null", "-", "--", "nan"]
for col in df2.select_dtypes(include="object").columns:
    df2[col] = df2[col].str.strip()
    df2[col] = df2[col].replace(replace_vals, np.nan)

# Kuppuswamy score stuff
kupp_map = {
    "Politician": 10, "Manager": 10, "Central Government Service": 10,
    "State Government Service": 10, "Tahsildar": 10,
    "Doctor": 9, "Doctor-Dentist": 9, "Doctor-General Physician": 9,
    "Doctor-Ophthalmologist": 9, "Doctor-Homeopathist": 9,
    "Doctor-Pediatrician": 9, "Doctor-Gynecologist": 9,
    "Doctor-Surgeon": 9, "Doctor-Neurologist": 9,
    "Doctor-Siddha": 9, "Doctor-Ayurvedic": 9,
    "Advocate": 9, "Architecture": 9, "AUDITOR": 9,
    "Engineer": 9, "IT Professional": 9,
    "Professor / Teacher / Education": 9, "Doctorate": 9,
    "Reporter": 8, "Accounts/Finance": 8, "Bank": 8,
    "IT Employee": 8, "Supervisor": 8, "Armed Forces": 8, "Police": 8,
    "Clerk": 7, "Government": 7,
    "Business": 6, "Self Employed": 6, "Fashion / Saloon": 6,
    "Retired Employee": 6, "Father In Church": 6, "Priest": 6, "Social Service": 6,
    "Farmer / Agriculture": 5,
    "Private Sector": 4,
    "Driver": 3, "Courier": 3,
    "Daily wages": 2,
    "Housewife": 1, "Retired": 1, "Armed Forces-Retired": 1, "Student": 1,
}
df2["Kupp_Occupation"] = df2["Occupation"].map(kupp_map)
df2 = df2.dropna(subset=["Kupp_Occupation"])

# Recode
df2["Gender"]      = df2["Gender"].map({"M": "Male", "F": "Female"})
df2["Gender_Code"] = df2["Gender"].map({"Male": 1, "Female": 2})

def age_group(age):
    if pd.isna(age):  return np.nan
    elif age < 30:    return 0
    elif age < 40:    return 1
    elif age < 50:    return 2
    elif age < 60:    return 3
    else:             return 4

age_labels = {0: "<30", 1: "30–39", 2: "40–49", 3: "50–59", 4: "≥60"}
df2["Age_Group"]     = df2["AGE"].apply(age_group)
df2["Age_Group_str"] = df2["Age_Group"].map(age_labels)
df2["SES_Group"]     = pd.cut(
    df2["Kupp_Occupation"], bins=[0, 4, 7, 10],
    labels=["Low (1–4)", "Middle (5–7)", "High (8–10)"]
)
df2["SES_Group_str"] = df2["SES_Group"].astype(str)
df2["Delta_HbA1c"]   = df2["HbA1c_FU"] - df2["HbA1c_BL"]
df2 = df2[df2["HbA1c_BL"].notna() & df2["HbA1c_FU"].notna()].copy()

print(f"Active final n: {len(df2)}")

# Load and preo inactive users 

df3 = pd.read_excel(
    "../mock-data/Active_Inactive_Non-Diala_Mock.xlsx",
    sheet_name="Inactive_Users"
)
df3 = df3.dropna(how="all")
print(f"\nInactive raw shape: {df3.shape}")

# Rename columns
df3 = df3.rename(columns={
    "HbA1c_baseline":  "HbA1c_BL",
    "HbA1c_followup":  "HbA1c_FU",
})

# Numeric coerce
df3["Diab_Duration"] = pd.to_numeric(df3["Diab_Duration"], errors="coerce")
for col in ["AGE", "AOS", "Diab_Duration", "HbA1c_BL", "HbA1c_FU"]:
    if col in df3.columns:
        df3[col] = pd.to_numeric(df3[col], errors="coerce")

# String clean
for col in df3.select_dtypes(include="object").columns:
    df3[col] = df3[col].str.strip()
    df3[col] = df3[col].replace(replace_vals, np.nan)

# Kuppuswamy stuff
df3["Kupp_Occupation"] = df3["Occupation"].map(kupp_map)
df3 = df3.dropna(subset=["Kupp_Occupation"])

# Recode
df3["Gender"]      = df3["Gender"].map({"M": "Male", "F": "Female"})
df3["Gender_Code"] = df3["Gender"].map({"Male": 1, "Female": 2})
df3["Age_Group"]     = df3["AGE"].apply(age_group)
df3["Age_Group_str"] = df3["Age_Group"].map(age_labels)
df3["SES_Group"]     = pd.cut(
    df3["Kupp_Occupation"], bins=[0, 4, 7, 10],
    labels=["Low (1–4)", "Middle (5–7)", "High (8–10)"]
)
df3["SES_Group_str"] = df3["SES_Group"].astype(str)
df3["Delta_HbA1c"]   = df3["HbA1c_FU"] - df3["HbA1c_BL"]
df3 = df3[df3["HbA1c_BL"].notna() & df3["HbA1c_FU"].notna()].copy()

print(f"Inactive final n: {len(df3)}")

# For ease, I built a combined dataframe here.

shared_cols = [
    "MRNO", "Gender", "Gender_Code",
    "AGE", "Age_Group", "Age_Group_str",
    "AOS", "Diab_Duration",
    "Kupp_Occupation", "SES_Group", "SES_Group_str",
    "HbA1c_BL", "HbA1c_FU", "Delta_HbA1c",
]

df_active   = df2[[c for c in shared_cols if c in df2.columns]].copy()
df_inactive = df3[[c for c in shared_cols if c in df3.columns]].copy()

df_active["Group"]   = "Active"
df_inactive["Group"] = "Inactive"

df_combined = pd.concat([df_active, df_inactive], ignore_index=True)

print(f"\nCombined shape: {df_combined.shape}")
print(f"Active n:   {(df_combined['Group'] == 'Active').sum()}")
print(f"Inactive n: {(df_combined['Group'] == 'Inactive').sum()}")
print(f"\nColumns: {list(df_combined.columns)}")
print(df_combined[["MRNO", "Group", "Gender", "AGE", "Kupp_Occupation",
                    "HbA1c_BL", "HbA1c_FU",
                    "Delta_HbA1c"]].head(10).to_string())

In [ ]:

# MODULE 1: DESCRIPTIVE COMPARISON — ACTIVE vs INACTIVE

print("=" * 70)
print("MODULE 1: DESCRIPTIVE COMPARISON — ACTIVE vs INACTIVE")
print("=" * 70)

active   = df_combined[df_combined["Group"] == "Active"]
inactive = df_combined[df_combined["Group"] == "Inactive"]

# Continuous variables 
continuous = {
    "Age (years)":               "AGE",
    "Age of Onset (years)":      "AOS",
    "Diabetes Duration (years)": "Diab_Duration",
    "Kuppuswamy SES Score":      "Kupp_Occupation",
    "HbA1c BL (%)":             "HbA1c_BL",
    "HbA1c FU (%)":             "HbA1c_FU",
    "Delta HbA1c":               "Delta_HbA1c",
}

print(f"\n{'Variable':<30} {'Active (Mean±SD)':>22} {'Inactive (Mean±SD)':>22} {'Active n':>10} {'Inactive n':>10}")
print("-" * 96)
for label, col in continuous.items():
    if col in df_combined.columns:
        a = active[col].dropna()
        i = inactive[col].dropna()
        a_str = f"{a.mean():.2f} ± {a.std():.2f}" if len(a) > 0 else "N/A"
        i_str = f"{i.mean():.2f} ± {i.std():.2f}" if len(i) > 0 else "N/A"
        print(f"{label:<30} {a_str:>22} {i_str:>22} {len(a):>10} {len(i):>10}")

# Categorical variables
print("\n" + "=" * 70)
print("CATEGORICAL COMPARISON")
print("=" * 70)

# Gender
print("\nGender:")
print(f"  {'':20} {'Active':>12} {'Inactive':>12}")
print(f"  {'-'*46}")
for val in ["Male", "Female"]:
    a_n   = (active["Gender"]   == val).sum()
    i_n   = (inactive["Gender"] == val).sum()
    a_pct = a_n / len(active)   * 100
    i_pct = i_n / len(inactive) * 100
    print(f"  {val:<20} {a_n:>5} ({a_pct:.1f}%)  {i_n:>5} ({i_pct:.1f}%)")

# Age Group
print("\nAge Group:")
print(f"  {'':20} {'Active':>12} {'Inactive':>12}")
print(f"  {'-'*46}")
for val in sorted(df_combined["Age_Group"].dropna().unique()):
    label = age_labels[val]
    a_n   = (active["Age_Group"]   == val).sum()
    i_n   = (inactive["Age_Group"] == val).sum()
    a_pct = a_n / len(active)   * 100
    i_pct = i_n / len(inactive) * 100
    print(f"  {label:<20} {a_n:>5} ({a_pct:.1f}%)  {i_n:>5} ({i_pct:.1f}%)")

# SES Group
print("\nSES Group:")
print(f"  {'':20} {'Active':>12} {'Inactive':>12}")
print(f"  {'-'*46}")
ses_order = ["Low (1–4)", "Middle (5–7)", "High (8–10)"]
for val in ses_order:
    a_n   = (active["SES_Group_str"]   == val).sum()
    i_n   = (inactive["SES_Group_str"] == val).sum()
    a_pct = a_n / len(active)   * 100
    i_pct = i_n / len(inactive) * 100
    print(f"  {val:<20} {a_n:>5} ({a_pct:.1f}%)  {i_n:>5} ({i_pct:.1f}%)")

In [ ]:

# MODULE 2: BASELINE HbA1c — ARE GROUPS DIFFERENT AT START?

print("=" * 70)
print("MODULE 2: BASELINE HbA1c COMPARISON — ACTIVE vs INACTIVE")
print("=" * 70)

a_bl = active["HbA1c_BL"].dropna()
i_bl = inactive["HbA1c_BL"].dropna()

print(f"\n  Active   n={len(a_bl)}  mean={a_bl.mean():.2f}  sd={a_bl.std():.2f}  "
      f"median={a_bl.median():.2f}")
print(f"  Inactive n={len(i_bl)}  mean={i_bl.mean():.2f}  sd={i_bl.std():.2f}  "
      f"median={i_bl.median():.2f}")

# Normality check 
_, p_a = stats.shapiro(a_bl.sample(min(len(a_bl), 5000), random_state=42))
_, p_i = stats.shapiro(i_bl.sample(min(len(i_bl), 5000), random_state=42))
print(f"\n  Shapiro-Wilk Active:   p={p_a:.4f} ({'normal' if p_a > 0.05 else 'non-normal'})")
print(f"  Shapiro-Wilk Inactive: p={p_i:.4f} ({'normal' if p_i > 0.05 else 'non-normal'})")

# Mann-Whitney U 
u_stat, u_p = stats.mannwhitneyu(a_bl, i_bl, alternative="two-sided")
u_sig = "***" if u_p < 0.001 else "**" if u_p < 0.01 else "*" if u_p < 0.05 else "ns"
print(f"\n  Mann-Whitney U={u_stat:.1f}  p={u_p:.4f}  {u_sig}")

if u_p < 0.05:
    print("  ► Groups differ significantly at baseline — must account for")
    print("    this when interpreting outcome differences in Module 3")
else:
    print("  ► Groups comparable at baseline — outcome differences in")
    print("    Module 3 more likely attributable to engagement")

# Box plot 
fig, ax = plt.subplots(figsize=(7, 5))
bp = ax.boxplot(
    [a_bl.values, i_bl.values],
    tick_labels=["Active", "Inactive"],
    patch_artist=True
)
colors = ["#5cb85c", "#d9534f"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_title("HbA1c at Baseline — Active vs Inactive",
             fontsize=13, fontweight="bold")
ax.set_ylabel("HbA1c (%)")
ax.set_xlabel("Group")

# Add significance annotation
y_max = max(a_bl.max(), i_bl.max()) + 0.5
ax.plot([1, 2], [y_max, y_max], color="black", linewidth=1)
ax.text(1.5, y_max + 0.1, u_sig, ha="center", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("comparison_module2_baseline_hba1c.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✔ Plot saved — open comparison_module2_baseline_hba1c.png")
print("\n✔ Module 2 complete — paste output and we move to Module 3.")

In [ ]:

# MODULE 3: DID HbA1c IMPROVE MORE IN ACTIVE vs INACTIVE?

print("=" * 70)
print("MODULE 3: DELTA HbA1c COMPARISON — ACTIVE vs INACTIVE")
print("=" * 70)
print("""
NOTE: Groups differed significantly at baseline (Module 2, p<0.001)
      Delta comparison should be interpreted with this in mind.
      A higher baseline in one group may naturally lead to larger drops.
""")

a_delta = active["Delta_HbA1c"].dropna()
i_delta = inactive["Delta_HbA1c"].dropna()

print(f"  Active   n={len(a_delta)}  mean={a_delta.mean():.3f}  "
      f"sd={a_delta.std():.3f}  median={a_delta.median():.3f}")
print(f"  Inactive n={len(i_delta)}  mean={i_delta.mean():.3f}  "
      f"sd={i_delta.std():.3f}  median={i_delta.median():.3f}")

# Direction of change 
print(f"\n  Active   delta: {'✓ improved (↓)' if a_delta.mean() < 0 else '✗ worsened (↑)'}")
print(f"  Inactive delta: {'✓ improved (↓)' if i_delta.mean() < 0 else '✗ worsened (↑)'}")

# Normality 
_, p_a = stats.shapiro(a_delta.sample(min(len(a_delta), 5000), random_state=42))
_, p_i = stats.shapiro(i_delta.sample(min(len(i_delta), 5000), random_state=42))
print(f"\n  Shapiro-Wilk Active:   p={p_a:.4f} ({'normal' if p_a > 0.05 else 'non-normal'})")
print(f"  Shapiro-Wilk Inactive: p={p_i:.4f} ({'normal' if p_i > 0.05 else 'non-normal'})")

# Mann-Whitney U 
u_stat, u_p = stats.mannwhitneyu(a_delta, i_delta, alternative="two-sided")
u_sig = "***" if u_p < 0.001 else "**" if u_p < 0.01 else "*" if u_p < 0.05 else "ns"
print(f"\n  Mann-Whitney U={u_stat:.1f}  p={u_p:.4f}  {u_sig}")

# Effect size (rank biserial correlation) 
n_a   = len(a_delta)
n_i   = len(i_delta)
rbc   = 1 - (2 * u_stat) / (n_a * n_i)
print(f"  Rank biserial correlation (effect size): r={rbc:.3f}")
print(f"  Interpretation: {'small' if abs(rbc) < 0.3 else 'medium' if abs(rbc) < 0.5 else 'large'} effect")

# Conclusion 
print(f"\n  ► ", end="")
if u_p < 0.05:
    if a_delta.mean() < i_delta.mean():
        print("Active users showed significantly greater HbA1c improvement than Inactive")
    else:
        print("Inactive users showed significantly greater HbA1c improvement than Active")
else:
    print("No significant difference in HbA1c improvement between Active and Inactive")

print(f"\n  ► IMPORTANT: Baseline HbA1c differed significantly between groups")
print(f"    Active BL:   {active['HbA1c_BL'].mean():.2f}%")
print(f"    Inactive BL: {inactive['HbA1c_BL'].mean():.2f}%")
print(f"    Interpret delta comparison cautiously — regression to mean")
print(f"    may partly explain differences if one group had higher baseline")

# Plot 
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: Delta comparison boxplot
ax = axes[0]
bp = ax.boxplot(
    [a_delta.values, i_delta.values],
    tick_labels=["Active", "Inactive"],
    patch_artist=True
)
colors = ["#5cb85c", "#d9534f"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
y_max = max(a_delta.max(), i_delta.max()) + 0.5
ax.plot([1, 2], [y_max, y_max], color="black", linewidth=1)
ax.text(1.5, y_max + 0.1, u_sig, ha="center", fontsize=12, fontweight="bold")
ax.set_title("Delta HbA1c — Active vs Inactive", fontweight="bold")
ax.set_ylabel("Δ HbA1c (FU - BL)")
ax.set_xlabel("Group")

# Plot 2: BL vs FU means side by side
ax = axes[1]
x     = np.array([0, 1])
a_means = [active["HbA1c_BL"].mean(),   active["HbA1c_FU"].mean()]
i_means = [inactive["HbA1c_BL"].mean(), inactive["HbA1c_FU"].mean()]
a_sds   = [active["HbA1c_BL"].std(),    active["HbA1c_FU"].std()]
i_sds   = [inactive["HbA1c_BL"].std(),  inactive["HbA1c_FU"].std()]

ax.errorbar(x - 0.05, a_means, yerr=a_sds, fmt="-o", color="#5cb85c",
            label="Active", linewidth=2, capsize=5, markersize=8)
ax.errorbar(x + 0.05, i_means, yerr=i_sds, fmt="-o", color="#d9534f",
            label="Inactive", linewidth=2, capsize=5, markersize=8)
ax.set_xticks([0, 1])
ax.set_xticklabels(["Baseline", "Follow-up"])
ax.set_title("HbA1c BL → FU — Active vs Inactive", fontweight="bold")
ax.set_ylabel("HbA1c (%)")
ax.legend()

plt.suptitle("HbA1c Comparison — Active vs Inactive Users",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("comparison_module3_delta_hba1c.png", dpi=150, bbox_inches="tight")
plt.close()

In [ ]:
# MODULE 4: FOLLOW-UP HbA1c — ARE GROUPS DIFFERENT AT END?

print("=" * 70)
print("MODULE 4: FOLLOW-UP HbA1c COMPARISON — ACTIVE vs INACTIVE")
print("=" * 70)

a_fu = active["HbA1c_FU"].dropna()
i_fu = inactive["HbA1c_FU"].dropna()

print(f"\n  Active   n={len(a_fu)}  mean={a_fu.mean():.2f}  sd={a_fu.std():.2f}  "
      f"median={a_fu.median():.2f}")
print(f"  Inactive n={len(i_fu)}  mean={i_fu.mean():.2f}  sd={i_fu.std():.2f}  "
      f"median={i_fu.median():.2f}")

# Normality 
_, p_a = stats.shapiro(a_fu.sample(min(len(a_fu), 5000), random_state=42))
_, p_i = stats.shapiro(i_fu.sample(min(len(i_fu), 5000), random_state=42))
print(f"\n  Shapiro-Wilk Active:   p={p_a:.4f} ({'normal' if p_a > 0.05 else 'non-normal'})")
print(f"  Shapiro-Wilk Inactive: p={p_i:.4f} ({'normal' if p_i > 0.05 else 'non-normal'})")

# Mann-Whitney U 
u_stat, u_p = stats.mannwhitneyu(a_fu, i_fu, alternative="two-sided")
u_sig  = "***" if u_p < 0.001 else "**" if u_p < 0.01 else "*" if u_p < 0.05 else "ns"
n_a    = len(a_fu)
n_i    = len(i_fu)
rbc    = 1 - (2 * u_stat) / (n_a * n_i)

print(f"\n  Mann-Whitney U={u_stat:.1f}  p={u_p:.4f}  {u_sig}")
print(f"  Rank biserial correlation (effect size): r={rbc:.3f}")
print(f"  Interpretation: {'small' if abs(rbc) < 0.3 else 'medium' if abs(rbc) < 0.5 else 'large'} effect")

# Conclusion 
print(f"\n  ► ", end="")
if u_p < 0.05:
    if a_fu.mean() < i_fu.mean():
        print("Active users had significantly lower HbA1c at follow-up than Inactive")
    else:
        print("Inactive users had significantly lower HbA1c at follow-up than Active")
else:
    print("No significant difference in follow-up HbA1c between Active and Inactive")

print(f"\n  ► Context (from Module 2 & 3):")
print(f"    Baseline differed: Active={active['HbA1c_BL'].mean():.2f}%  "
      f"Inactive={inactive['HbA1c_BL'].mean():.2f}%")
print(f"    Delta differed:    Active={active['Delta_HbA1c'].mean():.3f}  "
      f"Inactive={inactive['Delta_HbA1c'].mean():.3f}")
print(f"    Follow-up:         Active={a_fu.mean():.2f}%  Inactive={i_fu.mean():.2f}%")

# Plot 
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot 1: FU boxplot
ax = axes[0]
bp = ax.boxplot(
    [a_fu.values, i_fu.values],
    tick_labels=["Active", "Inactive"],
    patch_artist=True
)
colors = ["#5cb85c", "#d9534f"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
y_max = max(a_fu.max(), i_fu.max()) + 0.5
ax.plot([1, 2], [y_max, y_max], color="black", linewidth=1)
ax.text(1.5, y_max + 0.1, u_sig, ha="center", fontsize=12, fontweight="bold")
ax.set_title("HbA1c at Follow-up — Active vs Inactive", fontweight="bold")
ax.set_ylabel("HbA1c (%)")
ax.set_xlabel("Group")

# Plot 2: Full trajectory BL → FU for both groups
ax     = axes[1]
x      = np.array([0, 1])
a_bl_m = active["HbA1c_BL"].mean()
a_fu_m = active["HbA1c_FU"].mean()
i_bl_m = inactive["HbA1c_BL"].mean()
i_fu_m = inactive["HbA1c_FU"].mean()
a_bl_s = active["HbA1c_BL"].std()
a_fu_s = active["HbA1c_FU"].std()
i_bl_s = inactive["HbA1c_BL"].std()
i_fu_s = inactive["HbA1c_FU"].std()

ax.errorbar(x - 0.05, [a_bl_m, a_fu_m], yerr=[a_bl_s, a_fu_s],
            fmt="-o", color="#5cb85c", label="Active",
            linewidth=2, capsize=5, markersize=8)
ax.errorbar(x + 0.05, [i_bl_m, i_fu_m], yerr=[i_bl_s, i_fu_s],
            fmt="-o", color="#d9534f", label="Inactive",
            linewidth=2, capsize=5, markersize=8)

# Annotate deltas
ax.annotate(f"Δ={active['Delta_HbA1c'].mean():.2f}",
            xy=(1 - 0.05, a_fu_m), xytext=(0.6, a_fu_m + 0.3),
            fontsize=9, color="#5cb85c",
            arrowprops=dict(arrowstyle="->", color="#5cb85c"))
ax.annotate(f"Δ={inactive['Delta_HbA1c'].mean():.2f}",
            xy=(1 + 0.05, i_fu_m), xytext=(1.15, i_fu_m + 0.3),
            fontsize=9, color="#d9534f",
            arrowprops=dict(arrowstyle="->", color="#d9534f"))

ax.set_xticks([0, 1])
ax.set_xticklabels(["Baseline", "Follow-up"])
ax.set_title("HbA1c Trajectory BL → FU", fontweight="bold")
ax.set_ylabel("HbA1c (%)")
ax.legend()

plt.suptitle("Follow-up HbA1c Comparison — Active vs Inactive Users",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("comparison_module4_followup_hba1c.png", dpi=150, bbox_inches="tight")
plt.close()
print("\n✔ Plot saved — open comparison_module4_followup_hba1c.png")
print("\n✔ Module 4 complete — paste output and we move to Module 5.")

In [ ]:

# MODULE 5: SES AS CONFOUNDER — ACTIVE vs INACTIVE

print("=" * 70)
print("MODULE 5: SES COMPARISON & SES vs DELTA HbA1c — ACTIVE vs INACTIVE")
print("=" * 70)

ses_order = ["Low (1–4)", "Middle (5–7)", "High (8–10)"]

# 5a. Does SES differ between Active and Inactive? 
print("\n── 5a. SES Distribution — Active vs Inactive ────────────────────")

a_ses = active["Kupp_Occupation"].dropna()
i_ses = inactive["Kupp_Occupation"].dropna()

print(f"\n  Active   n={len(a_ses)}  mean={a_ses.mean():.2f}  sd={a_ses.std():.2f}  "
      f"median={a_ses.median():.2f}")
print(f"  Inactive n={len(i_ses)}  mean={i_ses.mean():.2f}  sd={i_ses.std():.2f}  "
      f"median={i_ses.median():.2f}")

u_stat, u_p = stats.mannwhitneyu(a_ses, i_ses, alternative="two-sided")
u_sig  = "***" if u_p < 0.001 else "**" if u_p < 0.01 else "*" if u_p < 0.05 else "ns"
n_a    = len(a_ses)
n_i    = len(i_ses)
rbc    = 1 - (2 * u_stat) / (n_a * n_i)
print(f"\n  Mann-Whitney U={u_stat:.1f}  p={u_p:.4f}  {u_sig}")
print(f"  Effect size (rank biserial): r={rbc:.3f}")

if u_p < 0.05:
    print("  ► SES differs significantly between groups — SES is a confounder")
    print("    Must account for SES when comparing HbA1c outcomes")
else:
    print("  ► SES comparable between groups — less likely to confound outcomes")

# 5b. SES group breakdown 
print("\n── 5b. SES Group Breakdown ──────────────────────────────────────")
print(f"\n  {'SES Group':<20} {'Active n (%)':>15} {'Inactive n (%)':>15}")
print(f"  {'-'*52}")
for grp in ses_order:
    a_n   = (active["SES_Group_str"]   == grp).sum()
    i_n   = (inactive["SES_Group_str"] == grp).sum()
    a_pct = a_n / len(active)   * 100
    i_pct = i_n / len(inactive) * 100
    print(f"  {grp:<20} {a_n:>5} ({a_pct:.1f}%)  {i_n:>5} ({i_pct:.1f}%)")

# Chi-square on SES group distribution
ct = pd.crosstab(df_combined["Group"], df_combined["SES_Group_str"])
ct = ct.reindex(columns=ses_order, fill_value=0)
chi2, chi_p, dof, _ = stats.chi2_contingency(ct)
chi_sig = "***" if chi_p < 0.001 else "**" if chi_p < 0.01 else "*" if chi_p < 0.05 else "ns"
print(f"\n  Chi-square on SES distribution: χ²={chi2:.3f}  df={dof}  "
      f"p={chi_p:.4f}  {chi_sig}")

# 5c. Does SES predict Delta HbA1c within each group? 
print("\n── 5c. Spearman: SES vs Delta HbA1c within each group ──────────")
for grp_name, grp_df in [("Active", active), ("Inactive", inactive)]:
    sub = grp_df[["Kupp_Occupation", "Delta_HbA1c"]].dropna()
    r, p = stats.spearmanr(sub["Kupp_Occupation"], sub["Delta_HbA1c"])
    sig  = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    print(f"\n  {grp_name}:  r={r:.3f}  p={p:.4f}  {sig}  (n={len(sub)})")
    if p < 0.05:
        direction = "higher SES → greater improvement" if r < 0 else "higher SES → less improvement"
        print(f"    ► {direction}")
    else:
        print(f"    ► No significant relationship between SES and HbA1c improvement")

#  5d. Delta HbA1c by SES group — Active vs Inactive 
print("\n── 5d. Delta HbA1c by SES Group ─────────────────────────────────")
print(f"\n  {'SES Group':<20} {'Active Delta':>15} {'Inactive Delta':>15}")
print(f"  {'-'*52}")
for grp in ses_order:
    a_delta = active[active["SES_Group_str"]   == grp]["Delta_HbA1c"].dropna()
    i_delta = inactive[inactive["SES_Group_str"] == grp]["Delta_HbA1c"].dropna()
    a_str = f"{a_delta.mean():.3f} ± {a_delta.std():.3f}" if len(a_delta) > 0 else "N/A"
    i_str = f"{i_delta.mean():.3f} ± {i_delta.std():.3f}" if len(i_delta) > 0 else "N/A"
    print(f"  {grp:<20} {a_str:>15} {i_str:>15}")

#  Plots 
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
colors_grp = {"Active": "#5cb85c", "Inactive": "#d9534f"}

# Plot 1: SES Score boxplot Active vs Inactive
ax = axes[0]
bp = ax.boxplot(
    [a_ses.values, i_ses.values],
    tick_labels=["Active", "Inactive"],
    patch_artist=True
)
for patch, color in zip(bp["boxes"], ["#5cb85c", "#d9534f"]):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
y_max = max(a_ses.max(), i_ses.max()) + 0.5
ax.plot([1, 2], [y_max, y_max], color="black", linewidth=1)
ax.text(1.5, y_max + 0.1, u_sig, ha="center", fontsize=12, fontweight="bold")
ax.set_title("SES Score — Active vs Inactive", fontweight="bold")
ax.set_ylabel("Kuppuswamy Score")
ax.set_xlabel("Group")

# Plot 2: SES group stacked bar Active vs Inactive
ax  = axes[1]
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
x   = np.arange(2)
bottom = np.zeros(2)
bar_colors = ["#d9534f", "#f0ad4e", "#5cb85c"]
for grp, color in zip(ses_order, bar_colors):
    vals = ct_pct[grp].values if grp in ct_pct.columns else np.zeros(2)
    ax.bar(["Active", "Inactive"], vals, bottom=bottom,
           label=grp, color=color, alpha=0.8)
    bottom += vals
ax.set_title("SES Group Distribution %", fontweight="bold")
ax.set_ylabel("Percentage (%)")
ax.legend(title="SES Group", loc="upper right")

# Plot 3: Delta HbA1c by SES group — Active
ax = axes[2]
data = [active[active["SES_Group_str"] == g]["Delta_HbA1c"].dropna().values
        for g in ses_order]
bp = ax.boxplot(data, tick_labels=ses_order, patch_artist=True)
for patch, color in zip(bp["boxes"], bar_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_title("Delta HbA1c by SES Group — Active", fontweight="bold")
ax.set_ylabel("Δ HbA1c (FU - BL)")
ax.set_xlabel("SES Group")
ax.tick_params(axis="x", rotation=15)

# Plot 4: Delta HbA1c by SES group — Inactive
ax = axes[3]
data = [inactive[inactive["SES_Group_str"] == g]["Delta_HbA1c"].dropna().values
        for g in ses_order]
bp = ax.boxplot(data, tick_labels=ses_order, patch_artist=True)
for patch, color in zip(bp["boxes"], bar_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.axhline(0, color="black", linestyle="--", linewidth=0.8, alpha=0.5)
ax.set_title("Delta HbA1c by SES Group — Inactive", fontweight="bold")
ax.set_ylabel("Δ HbA1c (FU - BL)")
ax.set_xlabel("SES Group")
ax.tick_params(axis="x", rotation=15)

plt.suptitle("SES Comparison & SES vs HbA1c Improvement — Active vs Inactive",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("comparison_module5_ses.png", dpi=150, bbox_inches="tight")
plt.close()